# Active Learning with `scikit-activeml`
**Adaptive Exam Preparation System — Production-Ready Implementation**

This notebook implements pool-based Active Learning using `scikit-activeml`.

| Strategy | scikit-activeml class | Method |
|---|---|---|
| **Uncertainty Sampling** | `UncertaintySampling` | `least_confident` |
| **Entropy-Based** | `UncertaintySampling` | `entropy` |
| **Query-by-Committee** | `QueryByCommittee` | vote entropy via BaggingClassifier |

**Stopping criterion:** Stop when no remaining question has `0.3 < P(correct) < 0.7`
(the CP is confident about every remaining question).(This will be handeled in the backend)

## 1. Setup & Imports

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import joblib
import json
import textstat
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    classification_report,
    ConfusionMatrixDisplay,
)

from skactiveml.pool import UncertaintySampling, QueryByCommittee
from skactiveml.classifier import SklearnClassifier

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Configuration

In [23]:
DIFFICULTY_MAP = {"easy": 1, "medium": 2, "hard": 3}

FEATURES = [
    "global_correctness",
    "topic_correctness",
    "easy_correct_avg",
    "medium_correct_avg",
    "hard_correct_avg",
    "avg_response_time",
    "questions_answered_so_far",
    "difficulty_encoded",
    "topic",
    "num_words",
    "qstn_complexity",
]

MODELS_DIR = Path("../models")
Data_dir= Path("../Dataset/clean")
MODELS_DIR.mkdir(exist_ok=True)

print(f"Features ({len(FEATURES)}): {FEATURES}")

Features (11): ['global_correctness', 'topic_correctness', 'easy_correct_avg', 'medium_correct_avg', 'hard_correct_avg', 'avg_response_time', 'questions_answered_so_far', 'difficulty_encoded', 'topic', 'num_words', 'qstn_complexity']


## 3. Load Pre-computed Artifacts from CP.ipynb

In [24]:
# ── Load train/test DataFrames (pre-computed by CP.ipynb) ────────────────────
train_df = pd.read_csv(str(Data_dir / "cp_train_df.csv"))
test_df = pd.read_csv(str(Data_dir / "cp_test_df.csv"))

# ── Load the scaler (fit by CP.ipynb) ───────────────────────────────────────
scaler = joblib.load(str(MODELS_DIR / "correctness_predictor_scaler.pkl"))

# ── Prepare feature matrices ────────────────────────────────────────────────
X_train = train_df[FEATURES]
y_train = train_df["correct"]
X_test = test_df[FEATURES]
y_test = test_df["correct"]

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

test_students = test_df["Student ID"].unique()

print(f" All artifacts loaded from CP.ipynb")
print(f"Train: {train_df['Student ID'].nunique()} students, {len(X_train)} rows")
print(f"Test : {len(test_students)} students, {len(X_test)} rows")
print(f"Scaler: {type(scaler).__name__}")

 All artifacts loaded from CP.ipynb
Train: 96 students, 13710 rows
Test : 25 students, 3630 rows
Scaler: StandardScaler


## 4. Load Correctness Predictor (from CP.ipynb)

In [29]:
# ── Load the saved CP model (Random Forest — best from CP.ipynb) ────────────
rf_base = joblib.load(str(MODELS_DIR / "correctness_predictor.pkl"))
print(f"Loaded CP model from CP.ipynb: {type(rf_base).__name__}")

# Wrap in SklearnClassifier for scikit-activeml
cp_model = SklearnClassifier(
    estimator=rf_base,
    classes=[0, 1],
    random_state=RANDOM_SEED,
)
cp_model.fit(X_train_scaled, y_train)

# Quick evaluation on test set
y_pred = rf_base.predict(X_test_scaled)
y_proba = rf_base.predict_proba(X_test_scaled)[:, 1]
print(f"CP Test Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"CP Test ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")

Loaded CP model from CP.ipynb: RandomForestClassifier
CP Test Accuracy : 0.837
CP Test ROC-AUC  : 0.883


## 5. Train Ensemble for Query-by-Committee

In [26]:
# BaggingClassifier for QBC — wraps diverse base estimators internally
bagging_base = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=6, class_weight="balanced", random_state=RANDOM_SEED
    ),
    n_estimators=10,
    max_samples=0.8,
    max_features=0.8,
    random_state=RANDOM_SEED,
)
bagging_base.fit(X_train_scaled, y_train)

# Wrap for scikit-activeml
ensemble_model = SklearnClassifier(
    estimator=BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=6, class_weight="balanced", random_state=RANDOM_SEED
        ),
        n_estimators=10,
        max_samples=0.8,
        max_features=0.8,
        random_state=RANDOM_SEED,
    ),
    classes=[0, 1],
    random_state=RANDOM_SEED,
)
ensemble_model.fit(X_train_scaled, y_train)

print("Ensemble model trained for QBC.")

Ensemble model trained for QBC.


## 8. Save All Production Artifacts

In [ ]:
# ── Note: CP model, scaler, and features are saved by CP.ipynb ──────────────
# ── We only save AL-specific artifacts here ──────────────────────────────────

import shutil

# ── Copy CP model as the AL correctness predictor ────────────────────────────
shutil.copy2(str(MODELS_DIR / "correctness_predictor.pkl"),
             str(MODELS_DIR / "al_correctness_predictor.pkl"))
print("Copied: correctness_predictor.pkl → al_correctness_predictor.pkl")

# ── Save QBC ensemble ───────────────────────────────────────────────────────
joblib.dump(bagging_base, str(MODELS_DIR / "al_ensemble_qbc.pkl"))
print("Saved: al_ensemble_qbc.pkl")

# ── Copy scaler ─────────────────────────────────────────────────────────────
shutil.copy2(str(MODELS_DIR / "correctness_predictor_scaler.pkl"),
             str(MODELS_DIR / "al_scaler.pkl"))
print("Copied: correctness_predictor_scaler.pkl → al_scaler.pkl")

# ── Save feature list ───────────────────────────────────────────────────────
with open(str(MODELS_DIR / "al_features.json"), "w") as f:
    json.dump(FEATURES, f, indent=2)
print("Saved: al_features.json")

# ── Save label encoders ─────────────────────────────────────────────────────
joblib.dump(le_student, str(MODELS_DIR / "al_student_label_encoder.pkl"))
joblib.dump(le_topic, str(MODELS_DIR / "al_topic_label_encoder.pkl"))
print("Saved: al_student_label_encoder.pkl")
print("Saved: al_topic_label_encoder.pkl")

Copied: correctness_predictor.pkl → al_correctness_predictor.pkl
Saved: al_ensemble_qbc.pkl
Copied: correctness_predictor_scaler.pkl → al_scaler.pkl
Saved: al_features.json
Saved: al_student_label_encoder.pkl
Saved: al_topic_label_encoder.pkl
Saved: al_skactiveml_uncertainty_sampling_results.csv
Saved: al_skactiveml_entropybased_results.csv
Saved: al_skactiveml_querybycommittee_results.csv
